# A1 Bear Camera AI — ONNX エクスポート (Colab)

このノートブックは Raspberry Pi 上の NCNN 推理が segfault する環境でも、
`models/yolo_bear.onnx` を作ることで ONNX Runtime ベースの安定推理に切り替えるための、
Pi 側への PyTorch/Ultralytics インストール不要のエクスポート手順です。

## 流れ

1. **Colab ランタイムを選択**（CPU で可）
2. リポジトリの `best.pt`（または `models/yolo_bear.pt`）を Colab にアップロード。
3. 既存の torch/torchvision は消さず、壊れている時だけ CPU 版 torch/torchvision を修復。
4. export 用の ultralytics / onnx / onnxscript / onnxslim を入れる。
5. `yolo_bear.onnx` を opset 18 でエクスポートし、可能なら opset 12 変換も試す（imgsz 256, simplify）。
6. ダウンロードして Raspberry Pi の `models/yolo_bear.onnx` に配置。

> **2026-07 更新**: `pip uninstall torch torchvision ...` は使わない。
> Colab の既存 torch を削除してから再インストールすると、torchvision 欠落や
> 大容量 CUDA wheel の再取得に入りやすい。今回のセルは torch/torchvision が
> import 不可の時だけ、CPU wheel の組み合わせ（torch 2.5.1 / torchvision 0.20.1）で修復する。
> ultralytics は Colab 既存の torch を使わせるため `--no-deps` で入れる。

安全性境界：この Camera AI はあくまで追加の感知レイヤであり、Arduino Uno Q の
接触パッド状態機械やフェイルセーフ RELEASE_OFF 動作を置き換えるものではない。

In [ ]:
import sys
print(sys.version)

In [ ]:
# Colab export dependency setup.
# Do not uninstall torch/torchvision: removing them can leave the runtime half-broken
# and can trigger very large CUDA wheel downloads. Repair only if imports are missing.
import importlib.util
import subprocess
import sys

def pip_install(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

def missing(module_name):
    return importlib.util.find_spec(module_name) is None

if missing('torch') or missing('torchvision'):
    print('torch/torchvision missing; installing matching CPU wheels...')
    pip_install(
        'torch==2.5.1',
        'torchvision==0.20.1',
        '--index-url',
        'https://download.pytorch.org/whl/cpu',
    )
else:
    print('using existing torch/torchvision')

# ONNX export tools can resolve their own lightweight dependencies.
pip_install('onnx==1.16.1', 'onnxruntime==1.27.0', 'onnxscript==0.5.7', 'onnxslim==0.1.94')

# Keep ultralytics from replacing the working Colab torch/torchvision stack.
pip_install('--no-deps', 'ultralytics==8.3.40', 'ultralytics-thop==2.0.9')

In [ ]:
import torch
import torchvision
import ultralytics
import onnx
import onnxruntime

print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)
print('onnx', onnx.__version__)
print('onnxruntime', onnxruntime.__version__)
print('cuda_available', torch.cuda.is_available())

## 1. best.pt をアップロード

Colab のファイルブラウザまたは次のセルで `best.pt` を `/content/best.pt` として用意する。

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # best.pt を選択
if not uploaded:
    raise RuntimeError('best.pt がアップロードされていません。')
name = next(iter(uploaded))
src = Path('/content') / name
dst = Path('/content/best.pt')
if src != dst:
    src.rename(dst)
print('model:', dst, dst.stat().st_size, 'bytes')

## 2. ONNX エクスポート

imgsz は学習と同じ 256。NCNN と同じ入力サイズを維持する。

> **2026-07 更新**: torch 2.5+ 系の dynamo ベース ONNX exporter を使う。
> opset 18 でエクスポートし、後続セルで opset 12 への変換を試みる。
> 変換に失敗しても Raspberry Pi 側は ONNX Runtime で opset 18 を実行できる。

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/best.pt', task='detect')
print('classes:', model.names)

# torch 2.5+ の dynamo exporter は opset >= 18 必須。
# 後続セルで opset 12 への変換も試す。失敗しても ONNX Runtime で実行できる。
export_path = model.export(
    format='onnx',
    imgsz=256,
    opset=18,
    simplify=True,
    dynamic=False,
    half=False,
    batch=1,
)
print('exported_onnx:', export_path)

In [ ]:
# opset 18 → 12 変換（古い cv2.dnn 互換が必要な場合の任意ステップ）
# モダン torch は opset 18 以上を出すことが多い。変換できなくても Pi 側 ONNX Runtime で実行できる。

import onnx
from onnx import version_converter
from pathlib import Path

m = onnx.load(export_path)
original_opset = m.opset_import[0].version
print(f'original opset: {original_opset}')

try:
    m12 = version_converter.convert_version(m, 12)
    export_path_12 = str(Path(export_path).with_suffix('.opset12.onnx'))
    onnx.save(m12, export_path_12)
    print(f'opset 12 converted: {export_path_12}')
    # 以降のセルで使うパスを opset 12 版に差し替え
    export_path = export_path_12
except Exception as e:
    print(f'opset conversion failed: {e}')
    print('opset 18 のまま進みます。Raspberry Pi では ONNX Runtime を使用します。')
    print('Pi 側は raspberry_pi/camera_ai/requirements.txt に含まれる onnxruntime で対応します。')

In [ ]:
import onnx
from pathlib import Path

export_path_str = export_path if isinstance(export_path, str) else str(export_path)
m = onnx.load(export_path_str)
opset = m.opset_import[0].version
print(f'file: {export_path_str}')
print(f'opset: {opset}')
inputs = [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in m.graph.input]
outputs = [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in m.graph.output]
print('inputs:', inputs)
print('outputs:', outputs)
assert inputs == [('images', [1, 3, 256, 256])], f'unexpected inputs: {inputs}'
assert outputs == [('output0', [1, 5, 1344])], f'unexpected outputs: {outputs}'
if opset <= 15:
    print('opset <= 15: cv2.dnn 互換の可能性あり。Pi runtime は ONNX Runtime を使用。')
else:
    print(f'opset {opset}: cv2.dnn ではなく Pi runtime の ONNX Runtime で実行します。')
print('✅ ONNX Runtime runtime contract OK')

## 3. yolo_bear.onnx にリネームしてダウンロード

出力を `yolo_bear.onnx` にしてダウンロード。Raspberry Pi 上で
`models/yolo_bear.onnx` として配置すると、Camera AI は自動で
ONNX Runtime ベースの推理に切り替わる（NCNN ディレクトリより優先）。

> opset 12 変換に成功した場合は変換済みファイルがダウンロードされる。
> 変換に失敗した場合でも opset 18 版がダウンロードされ、Pi 側の ONNX Runtime で実行できる。

In [ ]:
from pathlib import Path
from google.colab import files

src = Path(export_path) if isinstance(export_path, str) else Path(str(export_path))
final = Path('/content/yolo_bear.onnx')
if src != final:
    src.rename(final)
print('final:', final, final.stat().st_size, 'bytes')
files.download(str(final))

## 4. Raspberry Pi 側での配置

ダウンロードした `yolo_bear.onnx` を Pi 上の
`2026_Hackathon/models/yolo_bear.onnx` に置く。

```bash
# Pi 側
ls -la models/yolo_bear.onnx
```

その後、`./scripts/run_demo.sh` を起動すると、Camera AI は
ONNX Runtime 推理で動き、`ai_model_ok=true` となる。
NCNN が segfault 環境でも PyTorch 不要で安定動作する。

> **opset 変換失敗時の対応**: 変換に失敗して opset 18 のままでも問題ない。
> Pi 側の Camera AI は `raspberry_pi/camera_ai/requirements.txt` の `onnxruntime` を使う。